# Time Series Forecasting for Energy Analytics - NSP Dataset

## Objective
Build Prophet and LSTM forecasting models for all 5 regions using engineered features

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from prophet import Prophet
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Paths
ROOT_DIR = Path.cwd().parent
DATA_DIR = ROOT_DIR / "data"
OUTPUT_DIR = ROOT_DIR / "output"

# Helper functions
model_results = {'Model': [], 'Region': [], 'MAE': [], 'RMSE': [], 'MAPE': []}

def calculate_metrics(y_true, y_pred, model_name, region_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100

    model_results['Model'].append(model_name)
    model_results['Region'].append(region_name)
    model_results['MAE'].append(mae)
    model_results['RMSE'].append(rmse)
    model_results['MAPE'].append(mape)

    print(f"{model_name} - {region_name}: MAE={mae:.2f}, RMSE={rmse:.2f}, MAPE={mape:.2f}%")
    return mae, rmse, mape

def save_and_show_plot(fig, filename):
    output_path = OUTPUT_DIR / filename
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    print(f"Saved: {output_path}")
    plt.show()

## Load Engineered Features Dataset

In [ ]:
# Load engineered features with rolling statistics
df = pd.read_csv(OUTPUT_DIR / 'engineered_features.csv', parse_dates=['timestamp'])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")

# Verify key features exist
required_features = ['consumption_kwh_lag_1h', 'consumption_kwh_lag_24h',
                     'rolling_mean_168h', 'rolling_std_24h', 'hdd']

print("\nRequired features present:")
for feat in required_features:
    exists = feat in df.columns
    print(f"  {feat}: {'✓' if exists else '✗ MISSING'}")

# Handle region column (check if one-hot encoded or categorical)
if 'region' not in df.columns:
    region_cols = [col for col in df.columns if col.startswith('region_')]
    if region_cols:
        df['region'] = df[region_cols].idxmax(axis=1).str.replace('region_', '')
        print("\nReconstructed 'region' column from one-hot encoding")

print(f"\nRegions: {sorted(df['region'].unique())}")
print("\nSample data:")
df[['timestamp', 'region', 'consumption_kwh', 'consumption_kwh_lag_24h',
    'rolling_mean_168h', 'rolling_std_24h']].head()

## Prophet Forecasting - All Regions with Regressors

Features used as regressors:
- consumption_kwh (target)
- Lag features: lag_1h, lag_24h
- Rolling statistics: rolling_mean_168h, rolling_std_24h
- hdd (Heating Degree Days)
- is_holiday

We'll forecast each region separately with 80/20 train/test split.

In [ ]:
def forecast_region_prophet(region_name, df_full, max_years=None):
    """Run Prophet forecast for one region"""
    print(f"{'='*70}")
    print(f"Prophet Forecasting: {region_name}")
    print(f"{'='*70}")
    # Filter region
    region_df = df_full[df_full['region'] == region_name].copy()
    region_df = region_df.sort_values('timestamp').reset_index(drop=True)

    # Optionally limit to recent years to keep training time reasonable
    if max_years is not None and len(region_df) > 0:
        hours = int(24 * 365 * max_years)
        if len(region_df) > hours:
            print(f"Truncating to last {max_years} years ({hours} rows) for speed.")
            region_df = region_df.tail(hours).reset_index(drop=True)

    prophet_df = region_df[['timestamp', 'consumption_kwh', 
                            'temperature_c', 'humidity_pct', 
                            'hdd', 'consumption_kwh_lag_1h', 'consumption_kwh_lag_24h',
                            'rolling_mean_168h', 'rolling_std_24h',
                            'is_holiday']].copy()
    prophet_df.columns = ['ds', 'y', 'temperature', 'humidity', 'hdd',
                         'lag_1h', 'lag_24h', 'rolling_mean_168h', 'rolling_std_24h',
                         'is_holiday']

    # Train/test split (80/20)
    split_idx = int(len(prophet_df) * 0.8)
    train_df = prophet_df[:split_idx].copy()
    test_df = prophet_df[split_idx:].copy()

    print(f"Train: {len(train_df)} rows ({train_df['ds'].min()} to {train_df['ds'].max()})")
    print(f"Test:  {len(test_df)} rows ({test_df['ds'].min()} to {test_df['ds'].max()})")

    model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=True, seasonality_mode='multiplicative')
    model.add_regressor('temperature')
    model.add_regressor('humidity')
    model.add_regressor('hdd')
    model.add_regressor('lag_1h')
    model.add_regressor('lag_24h')
    model.add_regressor('rolling_mean_168h')
    model.add_regressor('rolling_std_24h')
    model.add_regressor('is_holiday')

    print('Training Prophet model...')
    model.fit(train_df)

    forecast = model.predict(test_df)

    calculate_metrics(test_df['y'].values, forecast['yhat'].values, 'Prophet', region_name)

    export_df = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    export_df['model'] = 'Prophet'
    export_df['region'] = region_name

    return export_df, model, forecast, test_df

In [ ]:
# Forecast all regions
regions = sorted(df['region'].unique())
print(f"Forecasting {len(regions)} regions: {regions}")

prophet_forecasts = []

for region in regions:
    # limit to last 3 years to keep runtime reasonable (adjustable)
    try:
        forecast_df, model, forecast, test_df = forecast_region_prophet(region, df, max_years=3)
        prophet_forecasts.append(forecast_df)
    except Exception as e:
        print(f"Error running Prophet for {region}: {e}")

# Combine
if prophet_forecasts:
    prophet_combined = pd.concat(prophet_forecasts, ignore_index=True)
    print(f"✓ Prophet forecasting complete: {len(prophet_combined)} predictions")
else:
    print('No Prophet forecasts generated.')

## LSTM Forecasting - Multi-Region with Multivariate Features

Features for LSTM:
- consumption_kwh (target)
- Lag features: lag_1h, lag_24h
- Rolling statistics: rolling_mean_168h, rolling_std_24h
- Weather: temperature_c, humidity_pct
- Time features: hour, day_of_week, is_weekend
- Grid features: grid_load_pct, renewable_pct

We'll use PyTorch LSTM with multivariate sequences.

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size, hidden_size=100, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        prediction = self.fc(last_output)
        return prediction

In [ ]:
def forecast_region_lstm(region_name, df_full, sequence_length=24, epochs=5):
    """Train LSTM for one region with multivariate features (quick run: epochs=5)"""
    print(f"\n{'='*70}")
    print(f"LSTM Forecasting: {region_name}")
    print(f"{'='*70}")
    
    # Filter and sort
    region_df = df_full[df_full['region'] == region_name].copy()
    region_df = region_df.sort_values('timestamp').reset_index(drop=True)
    
    # Ensure time features exist
    if 'hour' not in region_df.columns:
        region_df['hour'] = region_df['timestamp'].dt.hour
    if 'day_of_week' not in region_df.columns:
        region_df['day_of_week'] = region_df['timestamp'].dt.dayofweek
    if 'is_weekend' not in region_df.columns:
        region_df['is_weekend'] = region_df['day_of_week'].isin([5,6]).astype(int)
    
    # Select features
    feature_cols = ['consumption_kwh', 'consumption_kwh_lag_1h', 'consumption_kwh_lag_24h',
                   'rolling_mean_168h', 'rolling_std_24h',
                   'temperature_c', 'humidity_pct', 
                   'hour', 'day_of_week', 'is_weekend',
                   'grid_load_pct', 'renewable_pct']
    
    # Drop rows with missing features
    region_df = region_df.dropna(subset=feature_cols).reset_index(drop=True)
    if len(region_df) < sequence_length + 10:
        print('Not enough data after dropping NA for LSTM. Skipping.')
        return None, None, None, None
    
    data = region_df[feature_cols].values
    
    # Scale
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data)
    
    # Create sequences
    X, y = [], []
    for i in range(len(data_scaled) - sequence_length):
        X.append(data_scaled[i:i+sequence_length])
        y.append(data_scaled[i+sequence_length, 0])  # consumption_kwh is index 0
    
    X, y = np.array(X), np.array(y).reshape(-1, 1)
    
    # Train/test split (80/20)
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    print(f"Sequences: {len(X)}, Train: {len(X_train)}, Test: {len(X_test)}")
    print(f"Input shape: {X_train.shape}, Features: {len(feature_cols)}")
    
    # Convert to tensors
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    X_train_t = torch.FloatTensor(X_train).to(device)
    y_train_t = torch.FloatTensor(y_train).to(device)
    X_test_t = torch.FloatTensor(X_test).to(device)
    y_test_t = torch.FloatTensor(y_test).to(device)
    
    # Initialize model
    model = LSTMForecaster(input_size=len(feature_cols)).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    # Training
    print(f"Training on {device}...")
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 5 == 0:
            print(f"  Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        y_pred_t = model(X_test_t)
    
    # Inverse transform
    y_test_actual = scaler.inverse_transform(
        np.column_stack([y_test, np.zeros((len(y_test), len(feature_cols)-1))])
)[:, 0]
    y_pred_actual = scaler.inverse_transform(
        np.column_stack([y_pred_t.cpu().numpy(), np.zeros((len(y_pred_t), len(feature_cols)-1))])
)[:, 0]
    
    # Metrics
    calculate_metrics(y_test_actual, y_pred_actual, 'LSTM', region_name)
    
    return y_test_actual, y_pred_actual, model, scaler

In [ ]:
# Train LSTM for all regions (quick run: epochs=5)
lstm_results = {}

for region in regions:
    try:
        y_test, y_pred, model, scaler = forecast_region_lstm(region, df, epochs=5)
        if y_test is not None:
            lstm_results[region] = {'y_test': y_test, 'y_pred': y_pred}
    except Exception as e:
        print(f"Error training LSTM for {region}: {e}")

print(f"\n✓ LSTM forecasting quick run complete for {len(lstm_results)} regions")